In [1]:
# number of bootstrap replicates & parallel workers
num_replicates = 10        # ← set as desired
num_workers    = 4           # ← set to number of CPU cores you want to use

4

In [5]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "median_RV.csv"
    data_column                  = "x1"
    scale_multiplier             = 1.0
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 50
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "HAR"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "triweight"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "triweight"
    kernel_type_tvEWD            = "triweight"
    kernel_type_tvHAR            = "triweight"
    kernel_type_tvAR             = "triweight"

    alpha_level                  = 0.05
    verbose_output               = true
end

In [ ]:
using CSV, DataFrames, Random, Statistics, BSON
using Distributed

# launch worker processes
addprocs(num_workers)

# load essentials on each worker
@everywhere using Random, Statistics
@everywhere include("bootstrap_thresholds.jl")

      From worker 8:	WARNING: replacing module tvOLS_estimator.
      From worker 8:	WARNING: replacing module tvOLS_estimator.
      From worker 8:	WARNING: replacing module tvOLS_estimator.
      From worker 8:	WARNING: replacing module tvOLS_estimator.
      From worker 8:	WARNING: replacing module tvOLS_estimator.
      From worker 8:	WARNING: replacing module tvOLS_estimator.
      From worker 8:	WARNING: replacing module tvOLS_estimator.
      From worker 8:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing 

In [ ]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column)
series  = scale_multiplier .* Float64.(df[.!ismissing.(df[!, col_sym]), col_sym])

In [ ]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel_V2(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end

# remove working processes
rmprocs(workers())

In [ ]:
thr = compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

In [ ]:
# will create sed_thresholds.bson in your working directory
BSON.@save "sed_thresholds.bson" sed_vals thr